# Svara TTS Voice Clone — Inference Notebook

Zero-shot voice cloning for **19 Indic languages** using `kenpath/svara-tts-voiceclone-beta`.

- 🎤 Provide a short reference audio (~10s) — the model clones that voice
- 🗣️ Generate speech in any of 19 Indic languages with emotion/style tags
- ⚡ Optimized for Colab T4 (4-bit nf4, ~1.6 GB VRAM)
- 🧬 Based on Orpheus-style discrete audio tokens via SNAC codec

**References:**
- [Model on HF](https://huggingface.co/kenpath/svara-tts-voiceclone-beta)
- [Inference Server](https://github.com/Kenpath/svara-tts-inference)
- [Base Model](https://huggingface.co/kenpath/svara-tts-v1)

In [ ]:
%%capture
# ──────────────────────────────────────────────
# Install dependencies  (run once)
# ──────────────────────────────────────────────
!pip install -q torch==2.5.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu124
!pip install -q "transformers>=4.47.0,<5" accelerate>=1.2.0 bitsandbytes>=0.45.0
!pip install -q snac>=1.0.0 huggingface_hub>=0.27.0 sentencepiece>=0.2.0
!pip install -q soundfile>=0.13.0

print("✅ Dependencies installed")

In [ ]:
# ──────────────────────────────────────────────
# Imports & constants
# ──────────────────────────────────────────────
import torch, torchaudio, transformers, snac
import numpy as np, gc, warnings, io, time, re, os, logging
from io import BytesIO
from IPython.display import Audio, display
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("svara")

# ── Model IDs ──
MODEL_ID = "kenpath/svara-tts-voiceclone-beta"
CODEC_ID = "hubertsiuzdak/snac_24khz"
SAMPLE_RATE = 24000
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SNAC_DEVICE = DEVICE

# ── Token constants (match official svara-tts-inference) ──
TOKENISER_LENGTH = 128256
BOS_TOKEN       = 128000
END_OF_TURN     = 128009
AUDIO_TOKEN     = 156939  # <|audio|> marker for human turns
START_OF_SPEECH = 128257
END_OF_SPEECH   = 128258
START_OF_HUMAN  = 128259
END_OF_HUMAN    = 128260
START_OF_AI     = 128261
END_OF_AI       = 128262
PAD_TOKEN       = 128263
AUDIO_TOKENS_START = TOKENISER_LENGTH + 10          # 128266
AUDIO_VOCAB_SIZE   = 4096
AUDIO_TOKEN_OFFSETS = [AUDIO_TOKENS_START + i * AUDIO_VOCAB_SIZE for i in range(7)]

# ── GPU info ──
print(f"PyTorch: {torch.__version__}  |  CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}  |  VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.2f} GB")
print(f"Device: {DEVICE}  |  Audio offsets: {AUDIO_TOKEN_OFFSETS[0]}..{AUDIO_TOKEN_OFFSETS[-1]}")

In [ ]:
# ──────────────────────────────────────────────
# Audio utilities
# ──────────────────────────────────────────────

def load_audio_from_bytes(audio_bytes: bytes, device=None):
    """Load audio bytes → (waveform_tensor, sample_rate)."""
    if device is None: device = DEVICE
    buf = BytesIO(audio_bytes)
    w, sr = torchaudio.load(buf)
    if w.shape[0] > 1: w = w.mean(dim=0, keepdim=False)
    else: w = w.squeeze(0)
    return w.to(device=device, dtype=torch.float32), sr

def resample_audio(audio, orig_sr, target_sr, device=None):
    """Resample audio tensor to target sample rate."""
    if device is None: device = DEVICE
    if audio.dim() == 1: audio = audio.unsqueeze(0); sq = True
    else: sq = False
    r = torchaudio.transforms.Resample(orig_freq=orig_sr, new_freq=target_sr).to(device)
    audio = r(audio)
    return audio.squeeze(0) if sq else audio

def print_audio_stats(waveform, elapsed_sec, sr=SAMPLE_RATE):
    """Print duration / generation time / RTF."""
    d = len(waveform)/sr
    print(f"🔊 Audio: {d:.2f}s  |  Generation: {elapsed_sec:.2f}s  |  RTF: {elapsed_sec/d:.3f}x")

def save_audio(waveform, filename, sr=SAMPLE_RATE):
    """Save int16 numpy array to WAV."""
    torchaudio.save(filename, torch.from_numpy(waveform).float().unsqueeze(0) / 32767.0, sr)
    print(f"💾 Saved: {filename} ({len(waveform)/sr:.2f}s)")

print("✅ Utility functions loaded")

In [ ]:
# ──────────────────────────────────────────────────────────────────────
# Core classes: SNACCodec + HuggingFaceTransport + SvaraTTSOrchestrator
# ──────────────────────────────────────────────────────────────────────

class SNACCodec:
    """SNAC audio codec: encode waveform → interleaved 7-token frames, decode back."""

    def __init__(self, device="cuda"):
        self.device = device
        self.model = None

    def set_model(self, model):
        self.model = model

    def encode_audio(self, audio, input_sample_rate=24000, add_token_offsets=True):
        """
        Encode audio into interleaved 7-token-per-frame format.

        SNAC returns 3 codebooks with strides [4,2,1]:
          codes[0]: (N,)  — coarse frame
          codes[1]: (2N,) — medium
          codes[2]: (4N,) — fine

        We interleave as 7 tokens per coarse frame matching Svara vocabulary:
          [c0, c1, c2, c3, c4, c5, c6] = [codes[0][i], codes[1][2i], codes[2][4i],
                                            codes[2][4i+1], codes[1][2i+1],
                                            codes[2][4i+2], codes[2][4i+3]]
        """
        if audio.dim() == 1: audio = audio.unsqueeze(0).unsqueeze(0)
        elif audio.dim() == 2: audio = audio.unsqueeze(0)
        if input_sample_rate != 24000:
            audio = resample_audio(audio.squeeze(0), input_sample_rate, 24000)
            audio = audio.unsqueeze(0).unsqueeze(0)
        audio = audio.to(dtype=torch.float32, device=self.device)
        with torch.inference_mode():
            codes = self.model.encode(audio)
        c0 = codes[0].squeeze().cpu()  # (N,)
        c1 = codes[1].squeeze().cpu()  # (2N,)
        c2 = codes[2].squeeze().cpu()  # (4N,)
        n_frames = len(c0)
        all_tokens = []
        for i in range(n_frames):
            off = AUDIO_TOKEN_OFFSETS if add_token_offsets else [0]*7
            all_tokens.append(off[0] + c0[i].item())
            all_tokens.append(off[1] + c1[2*i].item())
            all_tokens.append(off[2] + c2[4*i].item())
            all_tokens.append(off[3] + c2[4*i+1].item())
            all_tokens.append(off[4] + c1[2*i+1].item())
            all_tokens.append(off[5] + c2[4*i+2].item())
            all_tokens.append(off[6] + c2[4*i+3].item())
        return all_tokens


class HuggingFaceTransport:
    """
    Transport layer: builds prompt as integer token IDs (bypassing string
    tokenization for special/audio tokens) and generates speech.
    """

    def __init__(self, model, tokenizer):
        self.model = model
        self.tokenizer = tokenizer

    def generate(self, text, audio_reference=None, temperature=0.75,
                 top_p=0.9, max_tokens=2048):
        """
        Build prompt token sequence and generate audio.

        Prompt format (matching official svara_text_to_tokens):
          BOS
          [START_OF_AI] [START_OF_SPEECH] <audio_tokens> [END_OF_SPEECH] [END_OF_AI] [END_OF_TURN]
          [START_OF_HUMAN] <text_token_ids> [END_OF_HUMAN] [END_OF_TURN]
          [START_OF_AI] [START_OF_SPEECH]
        """
        # Build token ID sequence directly (no string round-trip)
        input_ids = [BOS_TOKEN]

        if audio_reference is not None:
            # Reference audio as AI turn
            input_ids.extend([START_OF_AI, START_OF_SPEECH])
            input_ids.extend(audio_reference)
            input_ids.extend([END_OF_SPEECH, END_OF_AI, END_OF_TURN])

        # Target text as human turn (matches official _human_turn)
        #   [START_OF_HUMAN, AUDIO_TOKEN, <text>, END_OF_HUMAN, END_OF_TURN]
        text_ids = self.tokenizer.encode(text, add_special_tokens=False)
        input_ids.extend([START_OF_HUMAN, AUDIO_TOKEN])
        input_ids.extend(text_ids)
        input_ids.extend([END_OF_HUMAN, END_OF_TURN])

        # Final generation prefix (matches official _final_generation_prefix)
        #   [START_OF_AI, START_OF_SPEECH]
        input_ids.extend([START_OF_AI, START_OF_SPEECH])

        input_tensor = torch.tensor([input_ids], dtype=torch.long).to(self.model.device)

        # ── Generate ──
        with torch.inference_mode():
            output_ids = self.model.generate(
                input_tensor,
                max_new_tokens=max_tokens,
                temperature=temperature,
                top_p=top_p,
                do_sample=True,
                pad_token_id=PAD_TOKEN,
                eos_token_id=END_OF_SPEECH,
                use_cache=True,
            )

        # ── Extract generated audio tokens ──
        new_tokens = output_ids[0][input_tensor.shape[1]:]
        audio_tokens = []
        for tid in new_tokens:
            tid = tid.item()
            if tid == END_OF_SPEECH:
                break
            if tid >= AUDIO_TOKENS_START:
                audio_tokens.append(tid)

        if not audio_tokens:
            logger.warning("No audio tokens generated — returning silence")
            return np.zeros(SAMPLE_RATE // 10, dtype=np.int16)  # 0.1s silence

        return self._decode_tokens_to_pcm(audio_tokens)

    def _decode_tokens_to_pcm(self, tokens):
        """
        De-interleave 7-token frames back into 3 SNAC codebooks and decode.

        Inverse of encode_audio interleaving:
          frame[0] → codes[0][i]
          frame[1] → codes[1][2i]
          frame[2] → codes[2][4i]
          frame[3] → codes[2][4i+1]
          frame[4] → codes[1][2i+1]
          frame[5] → codes[2][4i+2]
          frame[6] → codes[2][4i+3]
        """
        codes_0, codes_1, codes_2 = [], [], []
        for i in range(0, len(tokens) - 6, 7):
            f = tokens[i:i+7]
            codes_0.append(f[0] - AUDIO_TOKEN_OFFSETS[0])
            codes_1.append(f[1] - AUDIO_TOKEN_OFFSETS[1])
            codes_2.append(f[2] - AUDIO_TOKEN_OFFSETS[2])
            codes_2.append(f[3] - AUDIO_TOKEN_OFFSETS[3])
            codes_1.append(f[4] - AUDIO_TOKEN_OFFSETS[4])
            codes_2.append(f[5] - AUDIO_TOKEN_OFFSETS[5])
            codes_2.append(f[6] - AUDIO_TOKEN_OFFSETS[6])

        t0 = torch.tensor(codes_0, dtype=torch.long, device=SNAC_DEVICE).unsqueeze(0)
        t1 = torch.tensor(codes_1, dtype=torch.long, device=SNAC_DEVICE).unsqueeze(0)
        t2 = torch.tensor(codes_2, dtype=torch.long, device=SNAC_DEVICE).unsqueeze(0)
        with torch.inference_mode():
            audio_hat = snac_model.decode([t0, t1, t2])
        if isinstance(audio_hat, list):
            audio_hat = audio_hat[0]
        pcm = audio_hat.squeeze().cpu().numpy()
        return np.clip(pcm * 32767, -32768, 32767).astype(np.int16)


class SvaraTTSOrchestrator:
    """High-level TTS orchestrator combining transport + codec."""

    def __init__(self, transport, model_id=MODEL_ID,
                 speaker_id="Telugu (Female)", device="cuda"):
        self.transport = transport
        self.model_id = model_id
        self.sample_rate = SAMPLE_RATE
        self.device = device
        self.speaker_id = speaker_id

    def warmup(self):
        """Warm up model + SNAC with a short dummy generation."""
        _ = self.transport.generate("warmup", audio_reference=None, max_tokens=1)
        print("✅ System ready — model, codec, and orchestrator loaded")

    def synthesize(self, text, audio_reference=None, temperature=0.75,
                   top_p=0.9, max_tokens=2048):
        """Synthesize speech from text, optionally conditioning on reference audio."""
        return self.transport.generate(
            text,
            audio_reference=audio_reference,
            temperature=temperature,
            top_p=top_p,
            max_tokens=max_tokens,
        )

print("✅ All classes defined")

In [ ]:
# ──────────────────────────────────────────────────────────────────────
# Load model (4-bit) + SNAC codec
# ──────────────────────────────────────────────────────────────────────

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
print(f"  Vocab size: {tokenizer.vocab_size}")

print("Loading model in 4-bit (nf4)...")
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    device_map="auto",
    low_cpu_mem_usage=True,
    attn_implementation="eager",  # T4 doesn't support flash_attention_2
)
print(f"  Model loaded ({sum(p.numel() for p in model.parameters())/1e9:.2f}B params)")

print("Loading SNAC codec...")
from snac import SNAC
snac_model = SNAC.from_pretrained(CODEC_ID).eval().to(SNAC_DEVICE)
snac_model = torch.compile(snac_model)
print(f"  SNAC quantizers: {len(snac_model.quantizer.quantizers)}")
print(f"  SNAC strides: {[q.stride for q in snac_model.quantizer.quantizers]}")

# ── Build pipeline ──
codec = SNACCodec(device=SNAC_DEVICE)
codec.set_model(snac_model)

transport = HuggingFaceTransport(model, tokenizer)
orchestrator = SvaraTTSOrchestrator(
    transport=transport,
    model_id=MODEL_ID,
    speaker_id="Telugu (Female)",
    device=SNAC_DEVICE,
)
orchestrator.warmup()

In [ ]:
# ──────────────────────────────────────────────────────────────────────
# Voice cloning helper
# ──────────────────────────────────────────────────────────────────────

def clone_voice(
    text: str = "",
    reference_audio_bytes: bytes = None,
    reference_audio_path: str = None,
    temperature: float = 0.75,
    top_p: float = 0.9,
    max_tokens: int = 2048,
):
    """
    Clone voice from reference audio and generate speech.

    Args:
        text: Target text to synthesize (supports <laugh>, <angry>, <sad>,
              <happy>, <neutral>, <excited>, <whisper>, <yawn> tags)
        reference_audio_bytes: Raw audio bytes for voice reference
        reference_audio_path: Path to audio file (alternative to bytes)
        temperature: Sampling temperature (0.5-1.0)
        top_p: Nucleus sampling threshold
        max_tokens: Maximum tokens to generate (~290 tokens ≈ 3s speech)

    Returns:
        (waveform: np.int16 array, elapsed_sec: float)
    """
    start = time.time()
    if not text:
        raise ValueError("`text` is required")

    audio_tokens = None
    if reference_audio_bytes is not None:
        audio_tensor, sr = load_audio_from_bytes(reference_audio_bytes, device=SNAC_DEVICE)
        duration = audio_tensor.shape[0] / sr
        logger.info(f"Reference audio: {duration:.1f}s @ {sr}Hz")
        if duration < 1.0:
            raise ValueError(f"Reference too short ({duration:.1f}s) — need ≥1.0s")
        local_codec = SNACCodec(device=SNAC_DEVICE)
        local_codec.set_model(snac_model)
        audio_tokens = local_codec.encode_audio(audio_tensor, input_sample_rate=sr, add_token_offsets=True)
        logger.info(f"Encoded {len(audio_tokens)} audio tokens ({len(audio_tokens)//7} frames)")
    elif reference_audio_path:
        with open(reference_audio_path, "rb") as f:
            return clone_voice(
                text=text,
                reference_audio_bytes=f.read(),
                temperature=temperature,
                top_p=top_p,
                max_tokens=max_tokens,
            )

    pcm = orchestrator.synthesize(
        text=text,
        audio_reference=audio_tokens,
        temperature=temperature,
        top_p=top_p,
        max_tokens=max_tokens,
    )
    elapsed = time.time() - start
    return np.frombuffer(pcm, dtype=np.int16), elapsed

print("✅ clone_voice() ready")
print()
print("Usage example:")
print('  waveform, elapsed = clone_voice("నమస్కారము", reference_audio_path="voice.wav")')
print("  display(Audio(waveform, rate=24000))")

In [ ]:
# ──────────────────────────────────────────────────────────────────────
# Run: download reference audio & clone voice
# ──────────────────────────────────────────────────────────────────────

import urllib.request

# Download a short reference audio clip (~10s, 24kHz Telugu speech)
REF_AUDIO_URL = "https://d.uguu.se/hscwVzYZ.wav"
REF_AUDIO_PATH = "/content/voice_ref.wav"

urllib.request.urlretrieve(REF_AUDIO_URL, REF_AUDIO_PATH)
print(f"📥 Downloaded reference: {os.path.getsize(REF_AUDIO_PATH)} bytes")

# ── Telugu text with emotion/style tags ──
telugu_text = (
    "<emotion:happy>నమస్కారము! నా పేరు స్వర.</emotion:happy> "
    "<emotion:neutral>నేను తెలుగులో మాట్లాడగలను.</emotion:neutral> "
    "<emotion:sad>ఈ రోజు చాలా విచారంగా ఉంది.</emotion:sad> "
    "<emotion:excited>ఇది చాలా అద్భుతమైన సాంకేతికత!</emotion:excited>"
)
print(f"📝 {telugu_text}")

# ── Clone ──
print("\n⏳ Generating voice clone (~2–4 min on T4)...")
waveform, elapsed = clone_voice(
    text=telugu_text,
    reference_audio_path=REF_AUDIO_PATH,
    temperature=0.75,
    top_p=0.9,
    max_tokens=2048,
)

print_audio_stats(waveform, elapsed)
display(Audio(waveform, rate=SAMPLE_RATE))

# Save output
output_path = "/content/cloned_voice.wav"
save_audio(waveform, output_path)
print(f"\n✅ Done — saved to {output_path}")